# CE310 Class 07 

**Building a Tree-based GP system from scratch: Operators and Interpreters**


This notebook builds on the prototype solutions of for Class 06  to produce a complete tree-based GP system.


**TASKS**

1. Create a notebook with the following code from the prototype solutions for last week's lab (see Moodle [https://moodle.essex.ac.uk/mod/folder/view.php?id=1422084])
```Python
from random import choice, random

# Primitives
functions = [['sin', 1], ['cos', 1], # Format: [Name, Arity]
            ['add', 2], ['sub', 2], ['mul', 2]] 

terminals = ['x', '1', '-1'] # Format: Name

PROB_TERM = len(terminals) / (len(terminals) + len(functions))

# Full initialisation
def full(maxD):
    # At max depth, pick a terminal
    if maxD == 0:
        return choice(terminals)
    else:
        # Pick a function and fill its arguments
        func, arity = choice(functions)
        expr = [func]
        for _ in range(arity):
            expr.append(full(maxD - 1))
        return expr

# Grow initialisation
def grow(maxD):
    # At max depth OR randomly decided to insert a terminal
    if maxD == 0 or random() < PROB_TERM:
        return choice(terminals)
    else:
        # Pick a function and fill its arguments
        func, arity = choice(functions)
        expr = [func]
        for _ in range(arity):
            expr.append(grow(maxD - 1))
        return expr
```

Also add the following tree-drawing function (which does not have external dependencies)
```Python
# INSTALL nltk AND svgling IF NEEDED

import nltk
from nltk import Tree
from IPython.display import display

def visualise_tree(prog):

    def visualise_tree_aux(prog):
        # If the prog is a list, it's a branch
        if isinstance(prog, (list, tuple)):
            label = str(prog[0])
            # Convert all remaining items in the list to children
            children = [visualise_tree_aux(child) for child in prog[1:]]
            return Tree(label, children)
        # Otherwise, it's a leaf
        else:
            return Tree(prog,[])
            
    display(visualise_tree_aux(prog))
```
Test that everything is working OK, by generating and drawing one tree for each initialisation method.

2. Now add the following interpreter adapted from the lecture notes:
```Python
from operator import add, sub, mul
from math import sin, cos

def interpreter(program, x):
    if program in terminals: # just one terminal
        # turn string into value
        return eval(program)
    else: # Larger program
        # find out function to invoke and its arguments
        func, arguments = program[0], program[1:]
        # Do recursion on arguments to get their value
        evaluated_arguments = [interpreter(arg, x) for arg in arguments]
        # Find implementation of function and call it with evaluated args
        evaluated_func = eval(func)
        return evaluated_func(*evaluated_arguments)
```

Test the interpreter with a program of your choice (e.g., random or say ```['sin', 'x']```) and different values of ```x```, to make sure it is working correctly.

3. Create a symbolic regression fitness function, i.e., a function that receives a program as input, then runs it for different values of ```x``` (in the range -5 to 5 in steps of 1), at each step comparing the output of the program ```y_prog``` with the value of ```y_target = (x - 3) ** 2```, via ```abs(y_prog - y_target)```, adding up such differences and turn turning the sum into a fintess value (higher fitness, greater similarity similarity between program outputs and targets).

Test with the following three programs:
```Python
prog1 = ['mul', ['sub', 'x', ['add', '1', '1']], ['sub', 'x', ['add', '1', '1']]]

prog2 = ['mul', ['sub', 'x', ['add', '1', 'x']], ['sub', 'x', ['add', '1', '1']]]

prog3 = ['mul', ['sub', 'x', ['add', 'x', 'x']], ['sub', 'x', ['add', '1', '1']]]
```

The first should produce the maximum fintess possible, the second should produce a worse fitness, the third the worst fitness of the three.

4. Now incorporate mutation from the lectures (adapted to avoid having to call ```mutate(deepcopy(parent))``` all the time to protect parents).
```Python
def mutate(prog):
        
    def mutate_aux(prog): 
        if isinstance(prog, list):
            mut_point = choice(range(1, len(prog)))
            if random() < 0.5:
                prog[mut_point] = grow(2)
            else:
                prog[mut_point] = mutate_aux(prog[mut_point])
        else:
            prog = grow(1)
        return prog
        
    return mutate_aux(deepcopy(prog))
```

Generate one program with the full method also evaluating its fitness. 

Use such program  as the first parent for a _(1 + 1) Evolution Strategy_ (i.e., mutate the parent, see if the offpring has better fitness, if not try again, else the offspring becomes the parent). 

Run it for 10,000 iterations and see how close the final solution is to the optimal fitness.

Compare the evolved program to the target function using this code
```Python
import matplotlib.pyplot as plt

plt.plot([(x - 2)**2 for x in range(-5, 6)],label='target')
plt.plot([interpreter(parent,x) for x in range(-5, 6)], label='best evolved')
plt.grid()
plt.legend()
plt.show()
```

5. Now transition to a generational GP system, adding crossover, truncation or tournament selection and a population. Crossover from the lecture notes (adapted) is provided below:
```Python
def get_random_subtree(prog):
    if not isinstance(prog, list) or random() < 0.5:
        # Return the current node (terminal or the whole subtree)
        return deepcopy(prog)
    else:
        # Randomly decide to go deeper into the tree
        idx = choice(range(1, len(prog)))
        return get_random_subtree(prog[idx])

def crossover(parent1, parent2):
    # We work on copies to avoid modifying the original population
    offspring = deepcopy(parent1)
    
    # If the offspring is a list, we can pick a sub-branch to replace
    if isinstance(offspring, list):
        # Pick a random point in parent 1 (skipping the operator at index 0)
        idx = choice(range(1, len(offspring)))
        
        if random() < 0.5: 
            replacement_subtree = get_random_subtree(parent2)
        else:
            replacement_subtree = crossover(offspring[idx], parent2)

        # Perform the swap
        offspring[idx] = replacement_subtree
        return offspring
    else:
        # If parent1 is just a terminal, crossover might just return offspring
        return offspring
```

For the rest of the elements, either implement them afresh or feel free to take inspiration from your solution to the Unit 5 lab (or my prototype solution see [https://moodle.essex.ac.uk/mod/folder/view.php?id=1335346])

Test with a population of 400 and 25 generations resulting in the same number of fitness evaluations as for the (1+1) ES. See if you get better results than then with the evolution strategy.